## Example 1: Basic logging with zero configuration

In [ ]:
from loguru import logger

# Example 1: Basic logging with zero configuration
print("Example 1: Basic Logging with Zero Configuration")

# Loguru works out of the box: its default handler writes to sys.stderr. logger.add() returns the
# integer id of a new handler; here a list sink captures the formatted records so they can be checked.
captured = []
sink_id = logger.add(captured.append, format="{level}|{message}|{extra}", level="DEBUG")
assert isinstance(sink_id, int), sink_id

print("Demonstrating basic logging levels:")
logger.debug("Debug message - detailed diagnostic info")
logger.info("Info message - general application flow")
logger.warning("Warning message - something unexpected happened")
logger.error("Error message - serious problem occurred")
logger.critical("Critical message - application may not continue")

# Keyword arguments fill {placeholders} in the message and are attached to the record's `extra` dict
logger.info("User action performed", user_id=12345, action="login")
logger.info("Operation completed: {count} items processed with status {status}", count=42, status="success")

# logger.exception() logs at ERROR level and appends the traceback
try:
    result = 10 / 0
except ZeroDivisionError:
    logger.exception("Division by zero occurred")
else:
    raise AssertionError(f"expected ZeroDivisionError, got {result!r}")

logger.remove(sink_id)

lines = [str(message) for message in captured]
levels = [line.split("|", 1)[0] for line in lines]
print(f"Captured {len(lines)} records at levels {levels}")
assert levels == ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL", "INFO", "INFO", "ERROR"], levels
assert lines[5].startswith("INFO|User action performed|{'user_id': 12345, 'action': 'login'}"), lines[5]
assert lines[6].startswith("INFO|Operation completed: 42 items processed with status success|"), lines[6]
assert "Traceback" in lines[7] and "ZeroDivisionError: division by zero" in lines[7], lines[7]

print("Basic logging checked")

## Example 2: File logging with rotation and filtering

In [ ]:
from loguru import logger
import tempfile
import os
import shutil

# Example 2: File logging with rotation and filtering
print("Example 2: File Logging with Rotation and Filtering")

# Create temporary directory for log files
temp_dir = tempfile.mkdtemp()
log_file = os.path.join(temp_dir, "app.log")

print(f"Setting up file logging to: {log_file}")

# Configure file handler with rotation
handler_id = logger.add(
    log_file,
    rotation="100 KB",  # Rotate when file reaches 100KB
    retention="10 days", # Keep logs for 10 days
    level="INFO",       # Only log INFO and above
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}",
    compression="zip"   # Compress rotated logs
)

print(f"File handler added with ID: {handler_id}")

# Test different log levels (only INFO+ will be written to file)
logger.debug("Debug message - should not appear in file")
logger.info("Info message - should appear in file")
logger.warning("Warning message - should appear in file")
logger.error("Error message - should appear in file")

# Add contextual information
for i in range(3):
    logger.info(f"Processing item {i}", item_id=i, batch="test_batch_001")

# Force flush and check file
logger.complete()  # Wait for all logging to complete

assert os.path.exists(log_file), log_file
with open(log_file, 'r') as f:
    content = f.read()
lines = content.strip().split('\n') if content.strip() else []
print(f"Log file contains {len(lines)} lines")

# Show first few lines
for i, line in enumerate(lines[:3]):
    print(f"  Line {i+1}: {line}")

# INFO, WARNING, ERROR and the three "Processing item" records; DEBUG is below the handler's level
assert len(lines) == 6, lines
assert "Debug message" not in content, content
assert lines[0].endswith("| INFO     | Info message - should appear in file"), lines[0]
assert lines[-1].endswith("| INFO     | Processing item 2"), lines[-1]

# Remove the handler to clean up
logger.remove(handler_id)
print(f"File handler removed")

# Show filtering and formatting options
print(f"\nLogging configuration options:")
options = [
    "Rotation: Size-based (100KB), time-based (daily), or count-based",
    "Retention: Automatic cleanup of old log files",
    "Compression: Automatic compression of rotated logs",
    "Filtering: Log level filtering per handler",
    "Custom formatting: Flexible message and timestamp formats"
]

for option in options:
    print(f"  • {option}")

# Cleanup
shutil.rmtree(temp_dir, ignore_errors=True)

print("File logging with rotation completed successfully!")

## Example 3: Structured logging with JSON serialization

In [ ]:
from loguru import logger
import json

# Example 3: Structured logging with JSON serialization
print("Example 3: Structured Logging with JSON Serialization")

# Custom serializer for structured logging
def json_serializer(record):
    subset = {
        "timestamp": record["time"].isoformat(),
        "level": record["level"].name,
        "message": record["message"],
        "module": record["module"],
        "function": record["function"],
        "line": record["line"]
    }
    
    # Add extra fields if present
    if record.get("extra"):
        subset.update(record["extra"])
        
    return json.dumps(subset)

# Add JSON handler
json_logs = []

def capture_json_log(message):
    json_logs.append(message.rstrip('\n'))

# A format= callable returns a format template that loguru then fills from the record; returning the rendered JSON
# itself would make loguru try to format the JSON braces and fail, so the JSON goes into record["extra"] and the
# template references it.
def json_format(record):
    record["extra"]["serialized"] = json_serializer(record)
    return "{extra[serialized]}\n"

handler_id = logger.add(
    capture_json_log,
    format=json_format,
    level="INFO"
)

print("JSON structured logging handler configured")

# Log structured data
logger.info("User login", user_id="user123", ip="192.168.1.1", success=True)
logger.info("Database query", table="users", query_time=0.045, rows_returned=25)
logger.warning("Rate limit exceeded", user_id="user456", requests=105, limit=100)
logger.error("Payment failed", order_id="ORD-789", amount=29.99, reason="insufficient_funds")

# Business event logging
logger.info("Order processed", 
            order_id="ORD-123",
            customer_id="CUST-456", 
            total_amount=149.99,
            items_count=3,
            payment_method="credit_card",
            shipping_method="express")

# Remove handler and show results
logger.remove(handler_id)

print(f"\nCaptured {len(json_logs)} structured log entries:")
expected_entries = [
    ("INFO", "User login", {"user_id": "user123", "ip": "192.168.1.1", "success": True}),
    ("INFO", "Database query", {"table": "users", "query_time": 0.045, "rows_returned": 25}),
    ("WARNING", "Rate limit exceeded", {"user_id": "user456", "requests": 105, "limit": 100}),
    ("ERROR", "Payment failed", {"order_id": "ORD-789", "amount": 29.99, "reason": "insufficient_funds"}),
    ("INFO", "Order processed", {"order_id": "ORD-123", "customer_id": "CUST-456", "total_amount": 149.99,
                                 "items_count": 3, "payment_method": "credit_card", "shipping_method": "express"}),
]
assert len(json_logs) == len(expected_entries), f"the JSON sink captured {len(json_logs)} of {len(expected_entries)} records"

for i, log_entry in enumerate(json_logs):
    try:
        parsed = json.loads(log_entry)
        print(f"Entry {i+1}:")
        print(f"  Timestamp: {parsed['timestamp']}")
        print(f"  Level: {parsed['level']}")  
        print(f"  Message: {parsed['message']}")
        
        # Show extra fields
        extra_fields = {k: v for k, v in parsed.items() 
                      if k not in ['timestamp', 'level', 'message', 'module', 'function', 'line']}
        if extra_fields:
            print(f"  Extra: {extra_fields}")
        print()
        assert (parsed['level'], parsed['message'], extra_fields) == expected_entries[i], parsed
    except json.JSONDecodeError as e:
        raise AssertionError(f"loguru emitted a structured-log line that is not JSON: {log_entry!r}") from e

# Show structured logging benefits
print("Structured logging benefits:")
benefits = [
    "Machine readable: Easy parsing and analysis",
    "Searchable fields: Query logs by specific attributes",
    "Consistent format: Uniform structure across application",
    "Rich context: Include relevant business data",
    "Integration ready: Works with log aggregation systems"
]

for benefit in benefits:
    print(f"  • {benefit}")

print("Structured logging completed successfully!")

## Example 4: Custom filters and conditional logging

In [ ]:
from loguru import logger
import time

# Example 4: Custom filters and conditional logging
print("Example 4: Custom Filters and Conditional Logging")

# Pass every ERROR/CRITICAL record; lower levels only when the record is bound with audit=True
def severity_or_audit_filter(record):
    return record["level"].no >= 40 or record["extra"].get("audit", False)

# Drop messages that mention sensitive information
def security_filter(record):
    message = record["message"].lower()
    sensitive_terms = ["password", "token", "secret", "key"]
    return not any(term in message for term in sensitive_terms)

filtered_logs, secure_logs, context_logs = [], [], []

handler_ids = [
    logger.add(lambda m: filtered_logs.append(m.rstrip("\n")), filter=severity_or_audit_filter,
               format="{level: <8} | {message}", level="DEBUG"),
    logger.add(lambda m: secure_logs.append(m.rstrip("\n")), filter=security_filter,
               format="{message}", level="INFO"),
    # Only records that carry a request_id (set by contextualize() below)
    logger.add(lambda m: context_logs.append(m.rstrip("\n")), filter=lambda record: "request_id" in record["extra"],
               format="{extra[request_id]} {extra[user_id]} | {message}", level="INFO"),
]
print("Custom filters configured")

print("\nTesting various log messages:")
logger.info("Application started successfully")
logger.warning("High memory usage detected: 85%")
logger.debug("Database connection pool status: 5/10 active")
logger.bind(audit=True).info("Configuration reloaded by admin")

# Error messages always pass the severity filter
logger.error("Database connection failed")
logger.critical("System shutdown initiated")

# Sensitive information is filtered out by the security filter
logger.info("User login attempt with password: secret123")
logger.info("API token generated: abc-123-def")
logger.info("Database connection string contains secret key")

# Context-aware logging: contextualize() adds fields to every record logged inside the block
with logger.contextualize(request_id="REQ-12345", user_id="USER-789"):
    logger.info("Processing user request")
    logger.warning("Slow query detected", query_time=2.5)

# Performance logging
start_time = time.perf_counter()
time.sleep(0.01)  # Simulate work
logger.info("Operation completed", duration_ms=round((time.perf_counter() - start_time) * 1000, 2), success=True)

for handler_id in handler_ids:
    logger.remove(handler_id)

print(f"\nSeverity/audit filter kept {len(filtered_logs)} records:")
for log in filtered_logs:
    print(f"  {log}")
assert filtered_logs == [
    "INFO     | Configuration reloaded by admin",
    "ERROR    | Database connection failed",
    "CRITICAL | System shutdown initiated",
], filtered_logs

print(f"\nSecurity filter kept {len(secure_logs)} records")
assert not any(term in log.lower() for log in secure_logs for term in ("password", "token", "secret")), secure_logs
assert "Application started successfully" in secure_logs and "Operation completed" in secure_logs, secure_logs

print(f"Context records: {context_logs}")
assert context_logs == [
    "REQ-12345 USER-789 | Processing user request",
    "REQ-12345 USER-789 | Slow query detected",
], context_logs

print("Custom filtering checked")

## Example 5: Advanced logging patterns and production deployment

In [ ]:
from loguru import logger
from functools import wraps
import os
import shutil
import sys
import tempfile
import time

# Example 5: Advanced logging patterns and production deployment
print("Example 5: Advanced Logging Patterns and Production Deployment")

# Production-style configuration: console output, an application log and an error-only log
def configure_production_logging(log_dir):
    logger.remove()  # remove the default handler

    logger.add(
        sys.stderr,
        format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>",
        level="INFO",
        colorize=True
    )
    logger.add(
        os.path.join(log_dir, "application.log"),
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} | {message}",
        level="INFO",
        rotation="10 MB",
        retention="30 days",
        compression="gz"
    )
    logger.add(
        os.path.join(log_dir, "errors.log"),
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} | {message} | {extra}",
        level="ERROR",
        rotation="5 MB",
        retention="90 days",
        backtrace=True,  # Include full traceback
        diagnose=True,   # Include variable values
        compression="gz"
    )

log_dir = tempfile.mkdtemp()
configure_production_logging(log_dir)
records = []
logger.add(lambda message: records.append(message.record), level="INFO")
print(f"Production logging configured in {log_dir}")

# 1. Performance monitoring decorator
def log_performance(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        logger.info(f"Starting {func.__name__}", function=func.__name__)
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"Failed {func.__name__}", function=func.__name__,
                         duration_ms=round((time.perf_counter() - start_time) * 1000, 2), error=str(e))
            raise
        logger.info(f"Completed {func.__name__}", function=func.__name__,
                    duration_ms=round((time.perf_counter() - start_time) * 1000, 2), success=True)
        return result
    return wrapper

# 2. Business event tracking
def track_business_event(event_type, **context):
    logger.info(f"Business event: {event_type}", event_type=event_type, **context)

# 3. Request correlation
def with_request_context(request_id, user_id=None):
    context = {"request_id": request_id}
    if user_id:
        context["user_id"] = user_id
    return logger.contextualize(**context)

print("\nDemonstrating advanced patterns:")

@log_performance
def build_report(n):
    return sum(range(n))

@log_performance
def parse_amount(text):
    return float(text)

assert build_report(100) == 4950
try:
    parse_amount("not-a-number")
except ValueError:
    pass
else:
    raise AssertionError("parse_amount should raise ValueError")

track_business_event("user_registration", user_id="USER-12345", email="user@example.com", source="web")
with with_request_context("REQ-789", "USER-456"):
    logger.info("Processing payment request")
    track_business_event("payment_processed", amount=99.99, method="credit_card")

# Restore the default configuration; removing the handlers also closes the log files
logger.remove()
logger.add(sys.stderr)

with open(os.path.join(log_dir, "application.log")) as f:
    app_log = f.read()
with open(os.path.join(log_dir, "errors.log")) as f:
    error_log = f.read()
shutil.rmtree(log_dir, ignore_errors=True)

print(f"\napplication.log: {len(app_log.splitlines())} lines, errors.log: {len(error_log.splitlines())} lines")
for expected in ("Starting build_report", "Completed build_report", "Failed parse_amount",
                 "Business event: user_registration", "Processing payment request", "Business event: payment_processed"):
    assert expected in app_log, (expected, app_log)
assert len(error_log.splitlines()) == 1 and "Failed parse_amount" in error_log, error_log
assert "could not convert string to float" in error_log, error_log  # the error= field from {extra}

completed = next(r for r in records if r["message"] == "Completed build_report")
assert completed["extra"]["success"] is True and completed["extra"]["function"] == "build_report", completed["extra"]
payment = next(r for r in records if r["message"] == "Business event: payment_processed")
assert payment["extra"] == {"request_id": "REQ-789", "user_id": "USER-456",
                            "event_type": "payment_processed", "amount": 99.99, "method": "credit_card"}, payment["extra"]

print("Advanced logging patterns checked")